In [1]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split

# ==========================================
# CHARGEMENT ET NORMALISATION (CORRECTION CRITIQUE)
# ==========================================

with open('data/processing/cifar-10/meta_animals.json', 'r') as f:
    meta_animals = json.load(f)

classes_animaux = meta_animals['label_names']
num_classes = len(classes_animaux)

train_archive = np.load('data/processing/cifar-10/train_animals.npz')
test_archive  = np.load('data/processing/cifar-10/test_animals.npz')

X_train_full = train_archive['X']
y_train_full = train_archive['y']
X_test_flat  = test_archive['X']
y_test       = test_archive['y']

# ÉTAPE CRITIQUE : normalisation 0-255 → 0.0-1.0
# Sans ça, les gradients sont instables et le modèle apprend 10x plus lentement
X_train_full = X_train_full.astype('float32') / 255.0
X_test_flat  = X_test_flat.astype('float32')  / 255.0

# Split train / validation
X_train_flat, X_val_flat, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2, random_state=42, stratify=y_train_full
)

# Reshape en images pour le CNN
X_train_img = X_train_flat.reshape(-1, 32, 32, 3)
X_val_img   = X_val_flat.reshape(-1, 32, 32, 3)
X_test_img  = X_test_flat.reshape(-1, 32, 32, 3)

print(f"Classes : {classes_animaux}")
print(f"Train : {X_train_img.shape} | Val : {X_val_img.shape} | Test : {X_test_img.shape}")
print(f"Pixels min/max après normalisation : {X_train_img.min():.2f} / {X_train_img.max():.2f}")

Classes : ['bird', 'cat', 'deer', 'dog', 'frog', 'horse']
Train : (24000, 32, 32, 3) | Val : (6000, 32, 32, 3) | Test : (6000, 32, 32, 3)
Pixels min/max après normalisation : 0.00 / 1.00


In [ ]:
# ==========================================
# DATA AUGMENTATION — API MODERNE tf.keras.layers
# On intègre l'augmentation DANS le modèle lui-même
# Avantage : s'applique uniquement pendant l'entraînement,
# jamais sur la validation → pas de corruption des métriques
# ==========================================

# Plus besoin d'ImageDataGenerator
# L'augmentation sera une couche du modèle (voir bloc 3 corrigé)
print("Augmentation intégrée au modèle — pas de générateur externe.")
print(f"Train : {X_train_img.shape} | Val : {X_val_img.shape}")

Générateur d'augmentation configuré.
Chaque image sera transformée aléatoirement à chaque epoch → variété artificielle.


In [ ]:
def build_cnn_ameliore(num_classes):
    model = models.Sequential(name="CNN_Ameliore_NeuralZOO")
    model.add(layers.Input(shape=(32, 32, 3)))
    
    # ---- AUGMENTATION (active seulement en training=True) ----
    model.add(layers.RandomFlip("horizontal"))
    model.add(layers.RandomTranslation(0.1, 0.1))
    model.add(layers.RandomRotation(0.05))
    model.add(layers.RandomZoom(0.1))
    
    # Bloc 1
    model.add(layers.Conv2D(32, (3, 3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(32, (3, 3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.2))
    
    # Bloc 2
    model.add(layers.Conv2D(64, (3, 3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(64, (3, 3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.3))
    
    # Bloc 3
    model.add(layers.Conv2D(128, (3, 3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(128, (3, 3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.4))
    
    # Bloc 4
    model.add(layers.Conv2D(256, (3, 3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Dropout(0.4))
    
    # GlobalAveragePooling
    model.add(layers.GlobalAveragePooling2D())
    
    # Classificateur
    model.add(layers.Dense(256, activation='relu',
                           kernel_regularizer=regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(num_classes, activation='softmax'))
    
    return model

cnn_model = build_cnn_ameliore(num_classes)
cnn_model.summary()

Model: "CNN_Ameliore_NeuralZOO"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 8, 8, 128)      │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 8, 8, 128)      │             

 Total params: 653,350 (2.49 MB)

 Trainable params: 651,430 (2.49 MB)

 Non-trainable params: 1,920 (7.50 KB)

In [4]:
from tensorflow.keras.optimizers import Adam

# ==========================================
# COMPILATION ET ENTRAÎNEMENT
# ==========================================

# Adam avec learning rate initial légèrement plus bas pour plus de stabilité
cnn_model.compile(
    optimizer=Adam(learning_rate=5e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callback 1 : EarlyStopping — arrête si val_loss ne s'améliore pas pendant 8 epochs
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

# Callback 2 : ReduceLROnPlateau — divise le LR par 2 si stagnation pendant 4 epochs
# C'est crucial : permet au modèle de "descendre plus finement" dans le creux de la loss
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)

print("Lancement de l'entraînement CNN amélioré...")

history_cnn = cnn_model.fit(
    datagen.flow(X_train_img, y_train, batch_size=64),  # Avec augmentation !
    epochs=60,
    validation_data=(X_val_img, y_val),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Lancement de l'entraînement CNN amélioré...
Epoch 1/60
375/375 ━━━━━━━━━━━━━━━━━━━━ 31s 77ms/step - accuracy: 0.2688 - loss: 1.9895 - val_accuracy: 0.1867 - val_loss: 2.1114 - learning_rate: 5.0000e-04
Epoch 2/60
375/375 ━━━━━━━━━━━━━━━━━━━━ 36s 96ms/step - accuracy: 0.3542 - loss: 1.7218 - val_accuracy: 0.2610 - val_loss: 2.8293 - learning_rate: 5.0000e-04
Epoch 3/60
375/375 ━━━━━━━━━━━━━━━━━━━━ 31s 83ms/step - accuracy: 0.3911 - loss: 1.5922 - val_accuracy: 0.3747 - val_loss: 1.6146 - learning_rate: 5.0000e-04
Epoch 4/60
375/375 ━━━━━━━━━━━━━━━━━━━━ 26s 69ms/step - accuracy: 0.4278 - loss: 1.4927 - val_accuracy: 0.3325 - val_loss: 2.1925 - learning_rate: 5.0000e-04
Epoch 5/60
375/375 ━━━━━━━━━━━━━━━━━━━━ 28s 74ms/step - accuracy: 0.4506 - loss: 1.4288 - val_accuracy: 0.4148 - val_loss: 1.7595 - learning_rate: 5.0000e-04
Epoch 6/60
375/375 ━━━━━━━━━━━━━━━━━━━━ 32s 84ms/step - accuracy: 0.4715 - loss: 1.3696 - val_accuracy: 0.2418 - val_loss: 4.1738 - learning_rate: 5.0000e-04
Epoch 7/

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# ANALYSE COLORIMÉTRIQUE — IMAGES MOYENNES PAR CLASSE
# Objectif : visualiser pourquoi chat et chien sont confondus
# Méthode : calculer l'image moyenne de chaque classe (moyenne pixel par pixel)
# ==========================================

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle("Images moyennes par classe (32x32x3)\nRevèle la signature colorimétrique de chaque animal",
             fontsize=13, fontweight='bold')

for idx, classe in enumerate(classes_animaux):
    ax = axes[idx // 3][idx % 3]
    
    # Sélectionner toutes les images de cette classe
    mask = y_train == idx
    images_classe = X_train_img[mask]
    
    # Image moyenne : moyenne arithmétique de chaque pixel sur toutes les images
    image_moyenne = images_classe.mean(axis=0)
    
    # Luminosité moyenne et std pour caractériser la classe
    lum_moy = image_moyenne.mean()
    
    ax.imshow(image_moyenne)
    ax.set_title(f"{classe}\n(n={mask.sum()}, lum={lum_moy:.3f})", fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('images_moyennes_par_classe.png', dpi=150, bbox_inches='tight')
plt.show()
print("→ Observez la similarité des palettes chat/chien : c'est le biais principal !")

In [ ]:
# ==========================================
# ANALYSE DES CANAUX RGB PAR CLASSE
# Histogrammes de distribution des couleurs
# ==========================================

fig, axes = plt.subplots(len(classes_animaux), 3, figsize=(14, 14))
fig.suptitle("Distribution des canaux RGB par classe", fontsize=13, fontweight='bold')
couleurs = ['red', 'green', 'blue']

for idx, classe in enumerate(classes_animaux):
    mask = y_train == idx
    images = X_train_img[mask]
    
    for c, (canal, couleur) in enumerate(zip(range(3), couleurs)):
        ax = axes[idx][c]
        ax.hist(images[:, :, :, canal].flatten(), bins=50,
                color=couleur, alpha=0.7, density=True)
        ax.set_title(f"{classe} — {couleur.upper()}", fontsize=9)
        ax.set_xlim(0, 1)
        ax.tick_params(labelsize=7)
        if c == 0:
            ax.set_ylabel(classe, fontsize=9)

plt.tight_layout()
plt.savefig('histogrammes_rgb_par_classe.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==========================================
# EXPLOITATION DES FEATURES ENGINEERED (fichier CSV collègue)
# On charge les features calculées et on les utilise pour un MLP hybride
# ==========================================

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report

# Chargement des features extraites par ton collègue
df_features = pd.read_csv('data/processing/cifar-10/train_animals_image_features_sample.csv')

print("Features disponibles dans le CSV :")
print(df_features.columns.tolist())
print(f"\nShape : {df_features.shape}")
print(df_features.describe())

In [ ]:
# ==========================================
# MLP HYBRIDE — utilise les features engineered du CSV
# Ces features (HOG, LBP, LAB, entropie, contraste...) capturent
# des informations que les pixels bruts ne donnent pas directement
# ==========================================

# Note : les features sample n'ont peut-être pas les labels associés directement
# On les utilise pour l'analyse exploratoire et, si possible, pour un modèle complémentaire

# Sélection des features numériques pertinentes
features_utiles = ['mean_r', 'mean_g', 'mean_b', 'std_r', 'std_g', 'std_b',
                   'lab_L', 'lab_a', 'lab_b', 'sharpness', 'entropy', 'edge_density']

X_features = df_features[features_utiles].values

scaler = StandardScaler()
X_features_scaled = scaler.fit_transform(X_features)

print(f"Shape features pour MLP hybride : {X_features_scaled.shape}")
print("Ces features encodent :")
print("  - mean_r/g/b : couleur moyenne → discrimine frog (vert) de horse (brun)")
print("  - std_r/g/b : variabilité de couleur → texture du pelage")
print("  - lab_L : luminosité → chiens/chats souvent similaires ici !")
print("  - sharpness : netteté des contours → formes animales")
print("  - entropy : complexité visuelle → plumage oiseau vs fourrure lisse")
print("  - edge_density : densité des bords → silhouette de l'animal")

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# ==========================================
# ÉVALUATION COMPLÈTE ET MATRICES DE CONFUSION
# ==========================================

def evaluer_modele_complet(model, X_val, X_test, y_val, y_test, nom, is_cnn=True):
    print(f"\n{'='*60}")
    print(f" ÉVALUATION FINALE : {nom}")
    print('='*60)
    
    # Prédictions
    y_pred_val  = np.argmax(model.predict(X_val,  verbose=0), axis=1)
    y_pred_test = np.argmax(model.predict(X_test, verbose=0), axis=1)
    
    # Rapport de classification (validation)
    print("\n--- VALIDATION ---")
    print(classification_report(y_val, y_pred_val, target_names=classes_animaux))
    
    print("--- TEST ---")
    print(classification_report(y_test, y_pred_test, target_names=classes_animaux))
    
    # Matrices de confusion côte à côte
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    for ax, y_pred, titre in [(ax1, y_pred_val, 'Validation'),
                               (ax2, y_pred_test, 'Test')]:
        cm = confusion_matrix(y_val if ax == ax1 else y_test, y_pred)
        
        # Normaliser pour voir les pourcentages
        cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        
        sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
                    xticklabels=classes_animaux,
                    yticklabels=classes_animaux,
                    ax=ax, vmin=0, vmax=1)
        ax.set_title(f'{nom} — {titre}\n(% sur vraie classe)')
        ax.set_ylabel('Vraie classe')
        ax.set_xlabel('Classe prédite')
    
    plt.tight_layout()
    plt.savefig(f'confusion_{nom.replace(" ", "_")}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Focus chat vs chien
    idx_chat = classes_animaux.index('cat')
    idx_chien = classes_animaux.index('dog')
    
    cm_test = confusion_matrix(y_test, y_pred_test)
    print(f"\n--- FOCUS BIAIS CHAT/CHIEN ---")
    print(f"Chats classifiés comme chiens : {cm_test[idx_chat][idx_chien]} / {cm_test[idx_chat].sum()}")
    print(f"Chiens classifiés comme chats : {cm_test[idx_chien][idx_chat]} / {cm_test[idx_chien].sum()}")

# Lancer l'évaluation
evaluer_modele_complet(cnn_model, X_val_img, X_test_img, y_val, y_test, "CNN Amélioré")